# Layer 2 - Classification & Anomaly Detection
**CMPE 255-01 | Group 7 | Fake Review Detection on Yelp**

**Contributor:** Disha Jadav

This notebook implements two parallel detection tracks using behavioural features from either reviewer-level or review-level data:

| Track | Paradigm | Algorithms |
|-------|----------|------------|
| **A** | Supervised Classification | Decision Tree, Random Forest, SVM, MLP |
| **B** | Unsupervised Anomaly Detection | Isolation Forest, Local Outlier Factor |

**Input files (auto-detected):** `reviewer_features.csv`, `reviewer_profiles.csv`, or `reviews_enriched.csv`  
**Label (Track A):** `spam_rate > 0.5` for reviewer-level, or `is_spam` for review-level

---
## 0. Setup & Imports

In [3]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os
import sys
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

try:
    import sklearn  # noqa: F401
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'scikit-learn'])

from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score
)

# Track A - Supervised
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

# Track B - Unsupervised
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42
os.makedirs('plots', exist_ok=True)


def get_group_stratified_split(X, y, groups, test_size=0.20, random_state=42):
    """Return a holdout split that is stratified by label and group-aware by user_id."""
    n_splits = max(2, int(round(1 / test_size)))
    sgkf = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    best_split = None
    best_gap = float('inf')

    for tr_idx, te_idx in sgkf.split(X, y, groups):
        current_ratio = len(te_idx) / len(y)
        gap = abs(current_ratio - test_size)
        if gap < best_gap:
            best_gap = gap
            best_split = (tr_idx, te_idx)

    if best_split is None:
        # Fallback: stratified-only split when group split is not feasible
        tr_idx, te_idx = train_test_split(
            np.arange(len(y)),
            test_size=test_size,
            stratify=y,
            random_state=random_state
        )
        return tr_idx, te_idx

    return best_split


print('All imports and helper functions loaded.')

All imports and helper functions loaded.


---
## 1. Load & Inspect Data

In [4]:
candidate_files = [
    'reviewer_features.csv',
    'reviewer_profiles.csv',
    'reviews_enriched.csv',
    'output_csv/reviewer_profiles.csv',
    'output_csv/reviews_enriched.csv',
]

data_file = next((p for p in candidate_files if Path(p).exists()), None)
if data_file is None:
    raise FileNotFoundError(
        'No input file found. Expected one of: ' + ', '.join(candidate_files)
    )

df = pd.read_csv(data_file)

if 'is_spam' in df.columns:
    data_mode = 'review-level'
elif 'spam_rate' in df.columns:
    data_mode = 'reviewer-level'
else:
    raise ValueError('Could not infer label column. Need is_spam or spam_rate.')

print(f'Loaded: {data_file}')
print(f'Data mode: {data_mode}')
print(f'Shape: {df.shape}')
print('\nColumn list:')
print(df.columns.tolist())
df.head()

Loaded: reviewer_features.csv
Data mode: reviewer-level
Shape: (260277, 9)

Column list:
['user_id', 'total_reviews', 'mean_rating', 'rating_std', 'mean_review_length', 'mean_exclamations', 'mean_caps_ratio', 'spam_rate', 'concentration']


,user_id,total_reviews,mean_rating,rating_std,mean_review_length,mean_exclamations,mean_caps_ratio,spam_rate,concentration
0,5044,1,1.00,0.0,36.00,0.0,0.010695,1.0,1.00
1,5045,1,1.00,0.0,248.00,0.0,0.017455,1.0,1.00
2,5046,4,3.25,0.5,45.25,0.0,0.030664,1.0,0.25
3,5047,1,5.00,0.0,233.00,2.0,0.025038,1.0,1.00
4,5048,1,5.00,0.0,152.00,2.0,0.022277,1.0,1.00


In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print('\nBasic statistics:')
df.describe()

---
## 2. Feature Engineering & Label Creation

In [5]:
if data_mode == 'reviewer-level':
    # Reviewer is spammer if > 50% of their reviews are spam
    df['is_spam_reviewer'] = (df['spam_rate'] > 0.5).astype(int)
    target_col = 'is_spam_reviewer'
else:
    target_col = 'is_spam'

print(f'Target column: {target_col}')
print('Label distribution:')
print(df[target_col].value_counts())
print(f'\nSpam ratio: {df[target_col].mean():.3f}')

Target column: is_spam_reviewer
Label distribution:
is_spam_reviewer
0    200082
1     60195
Name: count, dtype: int64

Spam ratio: 0.231


In [6]:
if data_mode == 'reviewer-level':
    # Logical feature groups to support both reviewer_features.csv and reviewer_profiles.csv
    reviewer_feature_candidates = {
        'total_reviews': ['total_reviews', 'review_count'],
        'mean_rating': ['mean_rating', 'avg_rating'],
        'rating_std': ['rating_std'],
        'mean_review_length': ['mean_review_length', 'avg_review_length'],
        'mean_exclamations': ['mean_exclamations', 'avg_exclamations'],
        'mean_caps_ratio': ['mean_caps_ratio', 'capital_ratio'],
        'concentration': ['concentration', 'max_seller_fraction'],
    }

    selected_cols = []
    for _, candidates in reviewer_feature_candidates.items():
        found = next((c for c in candidates if c in df.columns), None)
        if found is not None:
            selected_cols.append(found)

    FEATURE_COLS = selected_cols
else:
    # Review-level features expected in reviews_enriched.csv
    review_candidates = [
        'rating', 'review_length', 'word_count', 'exclamation_count',
        'question_count', 'capital_ratio', 'avg_word_length'
    ]
    FEATURE_COLS = [c for c in review_candidates if c in df.columns]

if len(FEATURE_COLS) < 4:
    raise ValueError(
        f'Not enough usable feature columns found. Got: {FEATURE_COLS}'
    )

X = df[FEATURE_COLS].copy()
y = df[target_col].to_numpy()
groups = df['user_id'].to_numpy() if 'user_id' in df.columns else np.arange(len(df))

# Fill any remaining NaN with column median
X.fillna(X.median(numeric_only=True), inplace=True)

print(f'Using {len(FEATURE_COLS)} features: {FEATURE_COLS}')
print(f'Feature matrix shape: {X.shape}')
X.describe()

Using 7 features: ['total_reviews', 'mean_rating', 'rating_std', 'mean_review_length', 'mean_exclamations', 'mean_caps_ratio', 'concentration']
Feature matrix shape: (260277, 7)


,total_reviews,mean_rating,rating_std,mean_review_length,mean_exclamations,mean_caps_ratio,concentration
count,260277.000000,260277.000000,260277.000000,260277.000000,260277.000000,260277.000000,260277.000000
mean,2.338270,3.934914,0.283645,99.980324,1.239993,0.030060,0.773151
std,4.496138,1.183343,0.546035,90.026375,2.313640,0.023382,0.325061
min,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.005076
25%,1.000000,3.500000,0.000000,40.000000,0.000000,0.019594,0.500000
50%,1.000000,4.000000,0.000000,75.000000,0.500000,0.025573,1.000000
75%,2.000000,5.000000,0.547723,131.000000,1.800000,0.034133,1.000000
max,197.000000,5.000000,2.828427,2706.000000,360.333333,1.000000,1.000000


### 2.1 Feature Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr = X.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, ax=ax,
            linewidths=0.5, square=True)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/feature_correlation.png', dpi=150)
plt.show()

### 2.2 Feature Distribution by Label

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLS):
    ax = axes[i]
    for label, color in [(0, '#2196F3'), (1, '#F44336')]:
        subset = df[df['is_spam_reviewer'] == label][col].dropna()
        # Clip extreme tails for readability
        p1, p99 = subset.quantile(0.01), subset.quantile(0.99)
        subset = subset.clip(p1, p99)
        ax.hist(subset, bins=40, alpha=0.6, color=color,
                label='Spam' if label == 1 else 'Legitimate', density=True)
    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=7)

# Hide unused subplot
axes[-1].set_visible(False)

fig.suptitle('Feature Distributions: Spam vs Legitimate Reviewers',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/feature_distributions.png', dpi=150)
plt.show()

---
## 3. Train / Test Split (Group-Aware + Stratified Holdout)

We use a **group-aware** and **label-stratified** holdout split targeting **80 / 20**.

- Stratification preserves the spam-to-legitimate ratio.
- Group-awareness prevents leakage by ensuring a `user_id` does not appear in both train and test.
- Implementation uses `StratifiedGroupKFold` and selects the split closest to 20% test size.

In [7]:
train_idx, test_idx = get_group_stratified_split(
    X=X,
    y=y,
    groups=groups,
    test_size=0.20,
    random_state=RANDOM_STATE
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y[train_idx]
y_test = y[test_idx]
g_train = groups[train_idx]
g_test = groups[test_idx]

print(f'Train size : {X_train.shape[0]:,} rows')
print(f'Test size  : {X_test.shape[0]:,} rows')
print(f'Unique train groups: {len(np.unique(g_train)):,}')
print(f'Unique test groups : {len(np.unique(g_test)):,}')
print(f'Group overlap      : {len(set(g_train).intersection(set(g_test)))}')
print(f'\nTrain spam ratio : {y_train.mean():.3f}')
print(f'Test  spam ratio : {y_test.mean():.3f}')

Train size : 208,222 rows
Test size  : 52,055 rows
Unique train groups: 208,222
Unique test groups : 52,055
Group overlap      : 0

Train spam ratio : 0.231
Test  spam ratio : 0.231


---
## 4. Track A – Supervised Classification

We evaluate four classifiers chosen to span the **interpretability–complexity spectrum**:

| Model | Rationale |
|---|---|
| **Decision Tree (DT)** | Fully interpretable; exposes rule-based patterns |
| **Random Forest (RF)** | Ensemble of trees; robust to noise and outliers |
| **Support Vector Machine (SVM)** | Effective in high-margin, moderate-dimensional spaces |
| **MLP Neural Network** | Captures non-linear feature interactions |

In [8]:
# ── Define classifiers ───────────────────────────────────────────────────────
classifiers = {
    'Decision Tree': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', DecisionTreeClassifier(
            max_depth=10,
            class_weight='balanced',
            random_state=RANDOM_STATE
        ))
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(
            n_estimators=200,
            max_depth=15,
            class_weight='balanced',
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ]),
    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(
            kernel='rbf',
            probability=True,
            class_weight='balanced',
            random_state=RANDOM_STATE
        ))
    ]),
    'MLP': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(
            hidden_layer_sizes=(128, 64, 32),
            activation='relu',
            max_iter=500,
            early_stopping=True,
            random_state=RANDOM_STATE
        ))
    ]),
}

print('Classifiers defined:', list(classifiers.keys()))

Classifiers defined: ['Decision Tree', 'Random Forest', 'SVM', 'MLP']


In [ ]:
# - Train & evaluate all classifiers ------------------------------------------
results = {}

for name, pipeline in classifiers.items():
    print('\n' + '=' * 55)
    print(f'  Training: {name}')
    print('=' * 55)

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred, average='binary')
    ap = average_precision_score(y_test, y_proba)

    results[name] = {
        'pipeline': pipeline,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'auc': auc,
        'f1': f1,
        'ap': ap,
    }

    print(f'  AUC-ROC  : {auc:.4f}')
    print(f'  F1-Score : {f1:.4f}')
    print(f'  Avg Prec : {ap:.4f}')
    print()
    print(
        classification_report(
            y_test,
            y_pred,
            target_names=['Legitimate', 'Spam']
        )
    )

print('\nAll classifiers trained.')


  Training: Decision Tree
  AUC-ROC  : 0.6828
  F1-Score : 0.4444
  Avg Prec : 0.3655

              precision    recall  f1-score   support

  Legitimate       0.88      0.48      0.62     40016
        Spam       0.31      0.78      0.44     12039

    accuracy                           0.55     52055
   macro avg       0.59      0.63      0.53     52055
weighted avg       0.75      0.55      0.58     52055


  Training: Random Forest
  AUC-ROC  : 0.6808
  F1-Score : 0.4395
  Avg Prec : 0.3713

              precision    recall  f1-score   support

  Legitimate       0.86      0.54      0.67     40016
        Spam       0.32      0.71      0.44     12039

    accuracy                           0.58     52055
   macro avg       0.59      0.63      0.55     52055
weighted avg       0.74      0.58      0.61     52055


  Training: SVM


### 4.1 Summary Table

In [ ]:
summary = pd.DataFrame({
    'Model': list(results.keys()),
    'AUC-ROC': [results[k]['auc'] for k in results],
    'F1-Score': [results[k]['f1'] for k in results],
    'Avg Precision': [results[k]['ap'] for k in results],
}).sort_values('AUC-ROC', ascending=False).reset_index(drop=True)

print('Track A – Performance Summary')
print(summary.to_string(index=False))

### 4.2 ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#1976D2', '#388E3C', '#F57C00', '#D32F2F']

for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})",
            color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curves – Track A Classifiers', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/roc_curves_trackA.png', dpi=150)
plt.show()

### 4.3 Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for (name, res), color in zip(results.items(), colors):
    prec, rec, _ = precision_recall_curve(y_test, res['y_proba'])
    ap = res['ap']
    ax.plot(rec, prec, label=f"{name} (AP={ap:.3f})",
            color=color, linewidth=2)

baseline = y_test.mean()
ax.axhline(y=baseline, color='k', linestyle='--', linewidth=1,
           label=f'Baseline (={baseline:.3f})')
ax.set_xlabel('Recall', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Precision-Recall Curves – Track A Classifiers',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/pr_curves_trackA.png', dpi=150)
plt.show()

### 4.4 Feature Importance (Random Forest)

In [ ]:
rf_model = results['Random Forest']['pipeline'].named_steps['clf']
importances = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(importances.index, importances.values,
               color=sns.color_palette('viridis', len(importances)))
ax.set_xlabel('Mean Decrease in Impurity', fontsize=11)
ax.set_title('Random Forest – Feature Importances', fontsize=13, fontweight='bold')

for bar, val in zip(bars, importances.values):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('plots/feature_importance_rf.png', dpi=150)
plt.show()

### 4.5 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legit', 'Spam'],
                yticklabels=['Legit', 'Spam'])
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

plt.suptitle('Confusion Matrices – Track A', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plots/confusion_matrices_trackA.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Track B – Unsupervised Anomaly Detection

Track B trains on **legitimate reviewers only** (`spam_rate ≤ 0.5`), which mirrors a realistic scenario where labelled spam data is unavailable. Models learn the distribution of normal behaviour; reviewers that deviate from this distribution are flagged as anomalies.

| Algorithm | Core Idea |
|---|---|
| **Isolation Forest** | Randomly partitions the feature space; anomalies require fewer splits to isolate |
| **LOF** | Compares a point's local density to that of its neighbours; sparsely-placed points are anomalies |

In [ ]:
# ── Prepare legitimate-only training set ─────────────────────────────────────
scaler_b = StandardScaler()

# Training data: legitimate reviewers only
legit_mask = (y_train == 0)
X_train_legit = X_train[legit_mask]
X_train_legit_scaled = scaler_b.fit_transform(X_train_legit)

# Test data: all reviewers (to evaluate detection)
X_test_scaled = scaler_b.transform(X_test)

print(f'Training on {X_train_legit.shape[0]:,} legitimate reviewers')
print(f'Evaluating on {X_test_scaled.shape[0]:,} test reviewers')
print(f'Test spam ratio: {y_test.mean():.3f}')

In [ ]:
# ── Isolation Forest ─────────────────────────────────────────────────────────

# contamination = expected fraction of anomalies in training data
# We use the spam rate observed in training labels as contamination estimate
contamination_rate = float(np.round(y_train.mean(), 2))
print(f'Contamination parameter set to: {contamination_rate}')

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=contamination_rate,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
iso_forest.fit(X_train_legit_scaled)

# Predict: +1 = inlier (legitimate), -1 = outlier (spam)
iso_raw    = iso_forest.predict(X_test_scaled)
iso_pred   = np.where(iso_raw == -1, 1, 0)  # convert to 0/1
iso_scores = -iso_forest.score_samples(X_test_scaled)  # higher = more anomalous

iso_auc = roc_auc_score(y_test, iso_scores)
iso_f1  = f1_score(y_test, iso_pred)
iso_ap  = average_precision_score(y_test, iso_scores)

print(f'\nIsolation Forest — AUC-ROC: {iso_auc:.4f}, F1: {iso_f1:.4f}, AP: {iso_ap:.4f}')
print(classification_report(y_test, iso_pred, target_names=['Legitimate', 'Spam']))

In [ ]:
# ── Local Outlier Factor ──────────────────────────────────────────────────────
# novelty=True allows LOF to be used in predict() mode on unseen test data

lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=contamination_rate,
    novelty=True,
    n_jobs=-1
)
lof.fit(X_train_legit_scaled)

lof_raw    = lof.predict(X_test_scaled)
lof_pred   = np.where(lof_raw == -1, 1, 0)
lof_scores = -lof.score_samples(X_test_scaled)

lof_auc = roc_auc_score(y_test, lof_scores)
lof_f1  = f1_score(y_test, lof_pred)
lof_ap  = average_precision_score(y_test, lof_scores)

print(f'LOF — AUC-ROC: {lof_auc:.4f}, F1: {lof_f1:.4f}, AP: {lof_ap:.4f}')
print(classification_report(y_test, lof_pred, target_names=['Legitimate', 'Spam']))

### 5.1 ROC Curves – Track B

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for scores, pred, name, color in [
    (iso_scores, iso_pred, 'Isolation Forest', '#7B1FA2'),
    (lof_scores, lof_pred, 'LOF',              '#00796B'),
]:
    fpr, tpr, _ = roc_curve(y_test, scores)
    auc = roc_auc_score(y_test, scores)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})',
            color=color, linewidth=2)

ax.plot([0,1],[0,1],'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curves – Track B (Anomaly Detection)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/roc_curves_trackB.png', dpi=150)
plt.show()

### 5.2 2-D Anomaly Score Scatter (PCA projection)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_test_2d = pca.fit_transform(X_test_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (scores, name) in zip(axes, [
    (iso_scores, 'Isolation Forest Anomaly Score'),
    (lof_scores, 'LOF Anomaly Score'),
]):
    sc = ax.scatter(X_test_2d[:, 0], X_test_2d[:, 1],
                    c=scores, cmap='RdYlGn_r',
                    s=6, alpha=0.5, rasterized=True)
    plt.colorbar(sc, ax=ax, label='Anomaly Score')
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

plt.suptitle('PCA Projection – Anomaly Scores on Test Set',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/anomaly_scores_pca.png', dpi=150)
plt.show()

---
## 6. Combined Comparison

In [ ]:
all_results = pd.DataFrame([
    {
        'Track': 'A – Supervised',
        'Model': name,
        'AUC-ROC': res['auc'],
        'F1-Score': res['f1'],
        'Avg Precision': res['ap'],
    }
    for name, res in results.items()
] + [
    {'Track': 'B – Unsupervised', 'Model': 'Isolation Forest',
     'AUC-ROC': iso_auc, 'F1-Score': iso_f1, 'Avg Precision': iso_ap},
    {'Track': 'B – Unsupervised', 'Model': 'LOF',
     'AUC-ROC': lof_auc, 'F1-Score': lof_f1, 'Avg Precision': lof_ap},
])

all_results = all_results.sort_values('AUC-ROC', ascending=False).reset_index(drop=True)
print('Full Model Comparison:')
print(all_results.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(all_results))
width = 0.28
track_colors = ['#1976D2' if 'A' in t else '#7B1FA2'
                for t in all_results['Track']]

b1 = ax.bar(x - width, all_results['AUC-ROC'], width,
            label='AUC-ROC', color='#42A5F5')
b2 = ax.bar(x,          all_results['F1-Score'], width,
            label='F1-Score', color='#66BB6A')
b3 = ax.bar(x + width,  all_results['Avg Precision'], width,
            label='Avg Precision', color='#FFA726')

ax.set_xticks(x)
ax.set_xticklabels(all_results['Model'], rotation=15, ha='right', fontsize=9)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('All Models – AUC-ROC, F1, Average Precision',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.axvline(x=3.5, color='gray', linestyle=':', linewidth=1.5)
ax.text(1.5, 1.02, 'Track A – Supervised', ha='center', fontsize=8, color='#1976D2')
ax.text(4.5, 1.02, 'Track B – Unsupervised', ha='center', fontsize=8, color='#7B1FA2')
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('plots/all_models_comparison.png', dpi=150)
plt.show()

---
## 7. Conclusions

Summarise findings here after running the notebook. Key points to address:

1. Which supervised classifier achieved the best AUC-ROC / F1?
2. How do unsupervised methods compare to supervised in terms of AUC (note: no labels used during training)?
3. Which behavioural features (from the RF importance plot) are most informative?
4. What are the implications for the full multi-signal pipeline?

In [ ]:
print('=== FINAL SUMMARY ===')
print(all_results[['Track', 'Model', 'AUC-ROC', 'F1-Score']].to_string(index=False))